# Phase 4: Base Feature Engineering (The Hybrid Workflow)

## The Hybrid System Architecture
In a standard pipeline,EDA is completed entirely before FE begins. However, because these datasets contain string objects (Dates and Times), i have to use a **Hybrid Workflow** to prepare the data for advanced statistical analysis:

1. **EDA Part 1 (Completed):** Dropped undersized datasets (< 500 rows), removed NaNs, and purged duplicates.
2. **Base FE (Current Phase):** Translating string objects into numerical categories.
3. **EDA Part 2 (Next Phase):** Returning to the EDA notebook to run correlation matrices, check target distributions, and remove noise using the newly engineered pure numbers.

## Scope of Feature Engineering
At this stage, I am strictly performing **Object Translation** with zero scaling or normalization. The advanced EDA phase requires raw, unmanipulated numbers to accurately detect outliers and baseline correlations. 

Furthermore, based on the raw structural matrix of these specific ICT patterns, we cannot manufacture additional meaningful numerical features without risking multicollinearity or future bias. Therefore, our Feature Engineering is strictly limited to extracting temporal context:

*   **Date Columns** $\rightarrow$ `DayOfWeek` (0 = Monday, 6 = Sunday)
*   **Time Columns** $\rightarrow$ `Hour` (0-23) and `Session` (0=Asia, 1=London, 2=NY, 3=Sydney/Late)

Once these objects are translated and dropped, all datasets will be 100% numerical and ready for the final advanced EDA audit.

In [1]:
import os
import glob
import pandas as pd

def map_session(hour):
    if pd.isna(hour): return -1
    if 0 <= hour < 7: return 0
    elif 7 <= hour < 13: return 1
    elif 13 <= hour < 20: return 2
    else: return 3

csv_files = glob.glob(os.path.join('../data/processed/', '*.csv'))

for file in csv_files:
    df = pd.read_csv(file)
    
    date_cols = [col for col in df.columns if col.endswith('_Date')]
    time_cols = [col for col in df.columns if col.endswith('_Time')]
    
    for d_col in date_cols:
        df[d_col.replace('_Date', '_DayOfWeek')] = pd.to_datetime(df[d_col]).dt.dayofweek
        
    for t_col in time_cols:
        hour_col = pd.to_datetime(df[t_col], format='%H:%M:%S', errors='coerce').dt.hour
        df[t_col.replace('_Time', '_Hour')] = hour_col
        df[t_col.replace('_Time', '_Session')] = hour_col.apply(map_session)
        
    df.drop(columns=date_cols + time_cols, inplace=True)
    df.to_csv(file, index=False)